In [1]:
import requests
from dotenv import load_dotenv
import os
load_dotenv()

API_KEY = os.getenv("NEWSAPI_KEY")

url = "https://newsapi.org/v2/everything"

params = {
    "q": "Microsoft OR MSFT",
    "language": "en",
    "sortBy": "publishedAt",
    "pageSize": 20,
    "apiKey": API_KEY
}

response = requests.get(url, params=params)
data = response.json()

articles = []

for article in data["articles"]:
    articles.append({
        "title": article["title"],
        "description": article["description"],
        "content": article["content"],
        "date": article["publishedAt"]
    })

print(articles[:3])

[{'title': 'Why did Microsoft increase Xbox Game Pass prices? #gaming', 'description': 'Xbox Game Pass price hikes and “value” messaging Microsoft recently increased the price of every Xbox Game Pass tier in October 2025 , and the feed includes a reminder of how the company justified the move. In the snippet, Microsoft says the increases were dr…', 'content': 'Microsoft recently increased the price of every Xbox Game Pass tier in October 2025, and the feed includes a reminder of how the company justified the move.\r\nIn the snippet, Microsoft says the increa… [+1204 chars]', 'date': '2026-04-19T00:02:14Z'}, {'title': 'Can you use generative AI in Family Court? The rules and risks – Ex-Files', 'description': "OPINION: I've found it useful and cheaper to use generative AI chatbots.", 'content': '2. Confidentiality needs to be upheld, as well as privacy suppression. You should not enter any information into an AI chatbot that is not already in the public domain. Do not enter private infor

In [24]:
from telethon import TelegramClient

api_id = os.getenv("TELEGRAM_API_ID")
api_hash = os.getenv("TELEGRAM_API_HASH")

client = TelegramClient("session", api_id, api_hash)

channels = [
    "@marketfeed",
    "@nse_stock_market",
    "@indianstockmarket",
]

telegram_messages =[]
async def main():
    all_messages = []

    for channel in channels:
        print(f"\nFetching from: {channel}")
        
        async for message in client.iter_messages(channel, limit=100):
            if message.text:
                all_messages.append({
                    "channel": channel,
                    "text": message.text,
                    "date": message.date
                })

    print(f"\nTotal messages fetched: {len(all_messages)}")

    # preview
    for msg in all_messages[:5]:
        print(msg)
    telegram_messages = all_messages
        
await client.start()
await main()


Fetching from: @marketfeed

Fetching from: @nse_stock_market

Fetching from: @indianstockmarket

Total messages fetched: 138
{'channel': '@marketfeed', 'text': 'OIL JUMPS, STOCKS WOBBLE AS MIDEAST CEASEFIRE HANGS IN THE BALANCE-RTRS [...](https://x.com/FirstSquawk/status/2046030080569376843)', 'date': datetime.datetime(2026, 4, 20, 0, 56, 17, tzinfo=datetime.timezone.utc)}
{'channel': '@marketfeed', 'text': 'RISING US–IRAN TENSIONS PUT SLIGHT PRESSURE ON THE JAPANESE YEN, NUDGING IT LOWER. [...](https://x.com/FirstSquawk/status/2046029551529275473)', 'date': datetime.datetime(2026, 4, 20, 0, 54, 17, tzinfo=datetime.timezone.utc)}
{'channel': '@marketfeed', 'text': 'TRUMP ANNOUNCES US SEIZURE OF IRANIAN SHIP IN GULF OF OMAN DURING BLOCKADE-CNBC [...](https://x.com/FirstSquawk/status/2046028936619143391)', 'date': datetime.datetime(2026, 4, 20, 0, 51, 32, tzinfo=datetime.timezone.utc)}
{'channel': '@marketfeed', 'text': 'US NAVY SEIZES IRANIAN-FLAGGED SHIP NEAR HORMUZ, TEHRAN VOWS SWIFT

In [26]:
from sentence_transformers import SentenceTransformer
import chromadb

model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()
collection = client.get_or_create_collection(name="financial_news")

docs = []
metadatas = []

for article in articles:
    docs.append(article["content"])
    metadatas.append({
        "source": "newsapi"
    })

for msg in telegram_messages:
    docs.append(msg["text"])
    metadatas.append({
        "source": "telegram"
    })

embeddings = model.encode(docs)

collection.add(
    documents=docs,
    embeddings=embeddings.tolist(),
    metadatas=metadatas,
    ids=[str(i) for i in range(len(docs))]
)

Loading weights: 100%|████████████████████████████| 103/103 [00:00<00:00, 2958.49it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [27]:
import yfinance as yf

ticker = yf.Ticker("MSFT")

data = ticker.history(period="5d")

latest = data.iloc[-1]

market_data = f"""
Open: {latest['Open']}
High: {latest['High']}
Low: {latest['Low']}
Close: {latest['Close']}
Volume: {latest['Volume']}
"""

In [28]:
def retrieve_context(query, collection, embed_model, n_results=5):

    query_embedding = embed_model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results
    )

    docs = results["documents"][0]

    return docs

In [29]:
def build_context(docs):

    context = ""

    for i, doc in enumerate(docs):
        context += f"{i+1}. {doc}\n"

    return context

In [36]:
def create_prompt(prediction_price, market_data, context):

    prompt = f"""
You are a financial market analyst.

Machine Learning Model Prediction:
Predicted next-day closing price: {prediction_price}

Recent Market Data (Yahoo Finance):
{market_data}

Relevant Market News and Discussions:
{context}

Task:
Using the model prediction, numerical market data, and relevant news,
explain the possible direction of the stock price (bullish or bearish).

Explain:
• Whether the prediction seems reasonable
• Which news or market signals support the movement
• Any risk factors
• mention news topic only if directly related to the mentioned stock

Keep the explanation concise and analytical.
"""

    return prompt

In [37]:
import google.generativeai as genai
load_dotenv()

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

llm = genai.GenerativeModel("gemini-2.5-flash")

def generate_reasoning(prompt):

    response = llm.generate_content(prompt)

    return response.text

In [38]:
from IPython.display import display, Markdown

query = "Microsoft stock outlook AI partnership"

prediction = "msft 422"

# Step 1 retrieve docs
docs = retrieve_context(query, collection, embed_model=model)

# Step 2 build context
context = build_context(docs)

# Step 3 build prompt
prompt = create_prompt(prediction, market_data, context)

# Step 4 generate reasoning
reasoning = generate_reasoning(prompt)

display(Markdown(reasoning))
# print(reasoning)

The model predicts a next-day closing price of 422 for MSFT, a slight decrease from the recent close of 422.79. This suggests a **neutral-to-bearish** direction for the stock in the short term.

**Reasonableness of Prediction:**
The prediction of 422 is very close to the previous day's close and falls within the stock's recent trading range (Low 420.69, High 431.58). This minor projected movement makes the prediction appear reasonable, indicating potential stability or a slight downward drift rather than a significant shift.

**Supporting Market Signals:**
The recent market data shows that MSFT opened at 424.82, climbed significantly to a high of 431.58, but ultimately closed lower at 422.79, very near its daily low of 420.69. This inability to hold earlier gains, coupled with closing below the open and near the daily low, suggests potential selling pressure or profit-taking by the end of the trading day. This pattern aligns with a slightly bearish outlook for the subsequent day, as predicted by the model.

**Relevant News:**
News item 1, "Xbox's chief content officer Matt Booty shares interesting behind-the-scenes stories on how Microsoft's gaming teams collaborate," is directly related to Microsoft. While this news highlights positive internal collaboration within Microsoft's gaming division, which could be fundamentally bullish long-term, it does not directly support the immediate slight bearish price prediction. The model's prediction seems to be more influenced by the recent trading dynamics than this specific piece of news.

**Risk Factors:**
*   **Broader Market Sentiment:** General market trends and performance of indices like the S&P 500 or Nasdaq could override individual stock movements.
*   **Continued Selling Pressure:** The observed profit-taking or selling pressure from the previous day might persist, leading to further declines.
*   **Technical Levels:** The stock could breach key support levels, accelerating a downward move, or fail to find new buying interest.
*   **AI Sector Dynamics:** While specific news items provided are general, overall sentiment regarding AI ethics, regulatory scrutiny, or competitive developments could indirectly impact a major AI player like Microsoft.